In [1]:
import sys
import os

# Force Spark to use the exact same Python that's running this notebook
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Week5-Superstore").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
# Setup
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType, StructType, StructField, StringType, IntegerType, DoubleType
 
spark = SparkSession.builder.appName("Week5-Superstore").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [2]:
df = spark.read.csv(
    r"C:\Users\Ritesh\OneDrive\Desktop\Celebal Excellence Intership\Week5_Data_cleaning_assignment\Data\Sample - Superstore.csv",
    header=True,
    inferSchema=True,
    quote='"',
    escape='"',
    multiLine=True,
)
 
print("Row count:", df.count())
df.printSchema()
df.show(5)

Row count: 9994
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+

In [3]:
#Q1:  What are the key limitations of traditional MapReduce that make Spark a preferred 
#choice for modern big data processing? 

# - MapReduce writes intermediate results to DISK after every Map/Reduce
#   phase, causing heavy I/O overhead between stages.
# - Poor support for ITERATIVE algorithms (ML, graph processing) since
#   every iteration re-reads data from disk instead of reusing it.
# - Rigid two-stage model (only Map -> Reduce), no support for complex
#   multi-step pipelines in a single job.
# - High latency, not suited for interactive queries or streaming.
# - Verbose programming model (separate Map/Reduce classes) vs Spark's
#   concise DataFrame/RDD API.

# Q2:  Explain how Spark uses In-Memory Computing to speed up iterative machine learning 
#algorithms compared to disk-based systems. 

# - Iterative ML algorithms (e.g. gradient descent, k-means) scan the
#   same dataset repeatedly across many iterations.
# - Disk-based systems re-read from disk every iteration -> slow.
# - Spark loads the dataset into RAM once using .cache()/.persist(),
#   so every later iteration reads directly from memory instead of disk.
# - RAM access is orders of magnitude faster than disk I/O, so total
#   runtime for iterative jobs drops dramatically (Spark benchmarks show
#   up to ~100x speedup vs Hadoop MapReduce for in-memory workloads).


df.cache()
df.count()   # triggers the cache
print("Is cached:", df.is_cached)


Is cached: True


In [4]:
#Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a 
#specific set of columns: user_id and transaction_date. 

before = df.count()
df_dedup = df.dropDuplicates(["Customer ID", "Order Date"])
after = df_dedup.count()
 
print(f"Rows before: {before}")
print(f"Rows after dropDuplicates(['Customer ID','Order Date']): {after}")
print(f"Duplicate combinations removed: {before - after}")

Rows before: 9994
Rows after dropDuplicates(['Customer ID','Order Date']): 4992
Duplicate combinations removed: 5002


In [5]:
#Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 
#'West' and then group by product_category to find the average sale_amount.
result_q4 = (
    df.filter(df.Region == "West")
      .groupBy("Category")
      .agg(F.avg("Sales").alias("avg_sale_amount"))
      .orderBy(F.desc("avg_sale_amount"))
)
result_q4.show()

+---------------+----------------+
|       Category| avg_sale_amount|
+---------------+----------------+
|     Technology|420.687532554257|
|      Furniture|357.302324611033|
|Office Supplies|116.422376910912|
+---------------+----------------+



In [6]:
#Q5: What is the difference between .na.drop() and .na.fill()? Provide a code 
#example of filling null values in a status column with the string 'Unknown'. 

# On the real column (will show 0 rows changed, confirming clean data)
null_count_before = df.filter(F.col("Ship Mode").isNull()).count()
df_filled = df.na.fill({"Ship Mode": "Unknown"})
print("Nulls in Ship Mode before fill:", null_count_before)
 
# Synthetic proof-of-concept with an injected null
demo_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("status", StringType(), True),
])
demo_data = [("O1", "Shipped"), ("O2", None), ("O3", "Delivered")]
demo_df = spark.createDataFrame(demo_data, demo_schema)
 
print("Before .na.fill():")
demo_df.show()
 
print("After .na.fill({'status': 'Unknown'}):")
demo_df.na.fill({"status": "Unknown"}).show()
 
# .na.drop() for comparison -- removes the row instead of filling it
print("After .na.drop() on the same demo data:")
demo_df.na.drop().show()


Nulls in Ship Mode before fill: 0
Before .na.fill():
+--------+---------+
|order_id|   status|
+--------+---------+
|      O1|  Shipped|
|      O2|     NULL|
|      O3|Delivered|
+--------+---------+

After .na.fill({'status': 'Unknown'}):
+--------+---------+
|order_id|   status|
+--------+---------+
|      O1|  Shipped|
|      O2|  Unknown|
|      O3|Delivered|
+--------+---------+

After .na.drop() on the same demo data:
+--------+---------+
|order_id|   status|
+--------+---------+
|      O1|  Shipped|
|      O3|Delivered|
+--------+---------+



In [7]:
#Q6: Write a query to find the total count of records for each city in a DataFrame, but only 
#for cities where the count is greater than 100. 
result_q6 = (
    df.groupBy("City")
      .count()
      .filter(F.col("count") > 100)
      .orderBy(F.desc("count"))
)
result_q6.show()


+-------------+-----+
|         City|count|
+-------------+-----+
|New York City|  915|
|  Los Angeles|  747|
| Philadelphia|  537|
|San Francisco|  510|
|      Seattle|  428|
|      Houston|  377|
|      Chicago|  314|
|     Columbus|  222|
|    San Diego|  170|
|  Springfield|  163|
|       Dallas|  157|
| Jacksonville|  125|
|      Detroit|  115|
+-------------+-----+



In [8]:
#Q7: How does the immutability of Spark DataFrames affect how you perform "data 
#cleaning" steps like dropping columns or renaming them? 

# - Spark DataFrames are IMMUTABLE - once created, they cannot be
#   changed in place.
# - Operations like dropping or renaming a column don't modify the
#   original DataFrame; they return a NEW DataFrame with the change.
# - You must reassign the result back to a variable:
#     df = df.drop("temp_col")
#     df = df.withColumnRenamed("old_name", "new_name")
# - Forgetting to reassign means the original df is left unchanged -
#   a common beginner mistake.
# - Immutability also enables lineage tracking, which Spark uses for
#   fault tolerance (it can recompute any DataFrame from its origin).
df_with_drop = df.drop("Postal Code")
 
print("Original df still has Postal Code column?:", "Postal Code" in df.columns)
print("New df_with_drop has Postal Code column?:", "Postal Code" in df_with_drop.columns)


Original df still has Postal Code column?: True
New df_with_drop has Postal Code column?: False


In [9]:
#Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 
#30 (inclusive) and the subscription is 'Premium'. 

demo_schema_q8 = StructType([
    StructField("customer", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("subscription", StringType(), True),
])
demo_data_q8 = [
    ("Alice", 25, "Premium"),
    ("Bob", 35, "Premium"),
    ("Carol", 18, "Basic"),
    ("Dan", 30, "Premium"),
    ("Eve", 17, "Premium"),
]
demo_df_q8 = spark.createDataFrame(demo_data_q8, demo_schema_q8)
 
result_q8 = demo_df_q8.filter(
    (demo_df_q8.age.between(18, 30)) & (demo_df_q8.subscription == "Premium")
)
result_q8.show()

+--------+---+------------+
|customer|age|subscription|
+--------+---+------------+
|   Alice| 25|     Premium|
|     Dan| 30|     Premium|
+--------+---+------------+



In [10]:
#Q9: When cleaning a dataset, why is it often better to handle null values before 
#performing mathematical aggregations like sum() or avg()?

# - sum() and avg() SILENTLY IGNORE nulls by default, which can skew
#   results in misleading ways.
# - avg() divides by the count of NON-NULL values only, not total rows,
#   which can misrepresent the true average if nulls actually meant
#   something specific (e.g. "0" or "missing data that matters").
# - Nulls can also propagate through arithmetic expressions
#   (col1 + col2 returns null if either operand is null).
# - Cleaning nulls first (drop or fill with a sensible default) makes
#   aggregation results predictable and consistent with business logic.


In [11]:
#Q10: Write the code to revise a column named raw_timestamp by casting it to a 
#TimestampType and renaming it to event_time.
df_ts = (
    df
    .withColumn("event_time", F.to_timestamp(F.col("Order Date"), "M/d/yyyy"))
    .drop("Order Date")
)
 
df_ts.select("Order ID", "event_time").show(5)
df_ts.printSchema()


+--------------+-------------------+
|      Order ID|         event_time|
+--------------+-------------------+
|CA-2016-152156|2016-11-08 00:00:00|
|CA-2016-152156|2016-11-08 00:00:00|
|CA-2016-138688|2016-06-12 00:00:00|
|US-2015-108966|2015-10-11 00:00:00|
|US-2015-108966|2015-10-11 00:00:00|
+--------------+-------------------+
only showing top 5 rows
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nul

In [12]:
#Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it 
#considered a wide transformation? 

# - A SHUFFLE redistributes data across partitions/nodes so all rows
#   sharing the same key (e.g. same groupBy key) end up on the same
#   partition together.
# - It involves writing intermediate data to disk and transferring it
#   over the network between executors - expensive (I/O + network).
# - NARROW transformation: each output partition depends on only ONE
#   input partition (e.g. filter, map) - no shuffle needed.
# - WIDE transformation: each output partition may depend on MULTIPLE
#   input partitions (e.g. groupBy, join, distinct) - requires a shuffle.
# - Grouping is wide because rows with the same key can originally live
#   on different machines; Spark must shuffle them together first.

df.groupBy("Category").agg(F.sum("Sales")).explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[Category#14], functions=[sum(Sales#17)])
   +- Exchange hashpartitioning(Category#14, 200), ENSURE_REQUIREMENTS, [plan_id=574]
      +- HashAggregate(keys=[Category#14], functions=[partial_sum(Sales#17)])
         +- InMemoryTableScan [Category#14, Sales#17]
               +- InMemoryRelation [Row ID#0, Order ID#1, Order Date#2, Ship Date#3, Ship Mode#4, Customer ID#5, Customer Name#6, Segment#7, Country#8, City#9, State#10, Postal Code#11, Region#12, Product ID#13, Category#14, Sub-Category#15, Product Name#16, Sales#17, Quantity#18, Discount#19, Profit#20], StorageLevel(disk, memory, deserialized, 1 replicas)
                     +- FileScan csv [Row ID#0,Order ID#1,Order Date#2,Ship Date#3,Ship Mode#4,Customer ID#5,Customer Name#6,Segment#7,Country#8,City#9,State#10,Postal Code#11,Region#12,Product ID#13,Category#14,Sub-Category#15,Product Name#16,Sales#17,Quantity#18,Discount#19,Profit#20] Batched: false

In [13]:
#Q12: Write a code snippet that identifies and removes rows where the email column 
#contains null values OR the username is an empty string.

demo_schema_q12 = StructType([
    StructField("email", StringType(), True),
    StructField("username", StringType(), True),
])
demo_data_q12 = [
    ("a@x.com", "alice"),
    (None, "bob"),
    ("c@x.com", ""),
    ("d@x.com", "dan"),
]
demo_df_q12 = spark.createDataFrame(demo_data_q12, demo_schema_q12)
 
print("Before cleaning:")
demo_df_q12.show()
 
result_q12 = demo_df_q12.filter(
    demo_df_q12.email.isNotNull() & (demo_df_q12.username != "")
)
print("After removing null email OR empty username:")
result_q12.show()
 



Before cleaning:
+-------+--------+
|  email|username|
+-------+--------+
|a@x.com|   alice|
|   NULL|     bob|
|c@x.com|        |
|d@x.com|     dan|
+-------+--------+

After removing null email OR empty username:
+-------+--------+
|  email|username|
+-------+--------+
|a@x.com|   alice|
|d@x.com|     dan|
+-------+--------+



In [14]:
#Q13: How do you use the .agg() function to calculate multiple statistics at once, such as 
#the min, max, and mean of the price column? 
result_q13 = df.agg(
    F.min("Sales").alias("min_price"),
    F.max("Sales").alias("max_price"),
    F.mean("Sales").alias("mean_price"),
)
result_q13.show()


+---------+---------+-----------------+
|min_price|max_price|       mean_price|
+---------+---------+-----------------+
|    0.444| 22638.48|229.8580008304938|
+---------+---------+-----------------+



In [15]:
#Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true 
#when your source data contains messy or inconsistent date formats? 

# - inferSchema=True makes Spark sample the data and GUESS column types
#   automatically.
# - If a date column has inconsistent formats mixed together
#   (e.g. 2023-01-05, 01/05/2023, Jan 5 2023), Spark's inference can:
#     1. Fail to recognize it as a date at all and fall back to STRING,
#        silently disabling date-based operations.
#     2. Or misclassify rows that don't match the sampled pattern,
#        turning them into NULLS during later parsing/casting.
# - This causes SILENT data loss/corruption - no error is thrown, but
#   downstream filtering, sorting, or date math becomes wrong.
# - Real example I hit: on the Superstore CSV, embedded quote marks in
#   Product Name (e.g. 14 7/8" x 11") broke default CSV parsing and
#   caused inferSchema to type Sales/Quantity/Discount as STRING
#   instead of numeric - exactly this kind of silent failure.
# - Safer approach: explicitly define the schema (StructType) or read
#   as string and parse deliberately with to_date()/to_timestamp() and
#   a known format string, so malformed rows are caught on purpose.


In [22]:
#Q15: Write a final processing pipeline that: 

#1. Filters out duplicates. 
#2. Fills null prices with 0. 
#3. Groups by store_id to calculate total revenue. 

result_q15 = (
    df
    .dropDuplicates()                                  # 1. remove duplicate rows
    .na.fill({"Sales": 0})                              # 2. fill null sales with 0
    .groupBy("State")                                   # 3. group by store/state
    .agg(F.sum("Sales").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
)
result_q15.show()




+--------------+------------------+
|         State|     total_revenue|
+--------------+------------------+
|    California|457687.63150000037|
|      New York|310876.27100000007|
|         Texas| 170188.0457999998|
|    Washington|138641.27000000002|
|  Pennsylvania|116511.91400000002|
|       Florida| 89473.70799999996|
|      Illinois| 80166.10099999997|
|          Ohio| 78258.13599999998|
|      Michigan| 76269.61400000003|
|      Virginia| 70636.72000000002|
|North Carolina|         55603.164|
|       Indiana|53555.359999999986|
|       Georgia| 49095.83999999999|
|      Kentucky| 36591.74999999999|
|    New Jersey| 35764.31199999999|
|       Arizona|         35282.001|
|     Wisconsin|32114.609999999997|
|      Colorado|         32108.118|
|     Tennessee|30661.872999999996|
|     Minnesota|29863.149999999994|
+--------------+------------------+
only showing top 20 rows


In [34]:
import os

output_path = r"C:\Users\Ritesh\OneDrive\Desktop\Celebal Excellence Intership\Week5_Data_cleaning_assignment\Output\Cleaned_Superstore.csv"



with open(output_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(columns)          # header row
    for row in rows:
        writer.writerow(row)

print("Saved cleaned dataset to:", output_path)
print("Rows written:", len(rows))

Saved cleaned dataset to: C:\Users\Ritesh\OneDrive\Desktop\Celebal Excellence Intership\Week5_Data_cleaning_assignment\Output\Cleaned_Superstore.csv
Rows written: 9994
